In [0]:
# ============================================================
# Configuration
# ============================================================

CATALOG = "worldbank_ai"

BRONZE_SCHEMA = "bronze"
SILVER_SCHEMA = "silver"

# UC Volume containing the manually uploaded GEP PDFs.
PDF_VOLUME_PATH = (
    "/Volumes/worldbank_ai/bronze/raw_documents/"
    "global_economic_prospects/"
)

# Final page-level parsed table.
TARGET_TABLE = (
    f"{CATALOG}.{SILVER_SCHEMA}.gep_parsed_pages"
)

# Expected GEP editions.
EXPECTED_REPORT_YEARS = [
    2022,
    2023,
    2024,
    2025,
    2026
]

print(f"PDF path: {PDF_VOLUME_PATH}")
print(f"Target:   {TARGET_TABLE}")

In [0]:
# ============================================================
# Imports
# ============================================================

import os
import re
import hashlib

from datetime import datetime, timezone

from pyspark.sql import functions as F
from pyspark.sql import types as T

In [0]:
# ============================================================
# Discover PDFs in the Unity Catalog Volume
# ============================================================
#
# dbutils.fs.ls() may return paths beginning with:
#
#     dbfs:/Volumes/...
#
# PyMuPDF cannot open that URI directly.
#
# PyMuPDF needs the local FUSE-mounted Volume path:
#
#     /Volumes/...
# ============================================================

pdf_files = [
    file_info
    for file_info in dbutils.fs.ls(PDF_VOLUME_PATH)
    if file_info.name.lower().endswith(".pdf")
]

print(f"PDF files discovered: {len(pdf_files)}")

for file_info in pdf_files:

    # Convert dbfs:/Volumes/... -> /Volumes/...
    local_path = file_info.path

    if local_path.startswith("dbfs:/Volumes/"):
        local_path = local_path.replace(
            "dbfs:/Volumes/",
            "/Volumes/",
            1
        )

    print(f"File:       {file_info.name}")
    print(f"DBFS path:  {file_info.path}")
    print(f"Local path: {local_path}")
    print("-" * 70)

In [0]:
# ============================================================
# Validate expected document count
# ============================================================

if len(pdf_files) != 5:
    raise RuntimeError(
        f"Expected 5 GEP PDFs, found {len(pdf_files)}."
    )

print("GEP PDF count validation passed.")

In [0]:
# Run only if PyMuPDF is unavailable.
%pip install pymupdf

In [0]:
# ============================================================
# Load PDF parsing library
# ============================================================
#
# PyMuPDF gives us:
#   - page-level extraction
#   - text blocks
#   - page dimensions
#
# Later we can add specialized table/figure routing without
# changing the rest of the pipeline.
# ============================================================

try:
    import fitz

    print(
        f"PyMuPDF available. Version: "
        f"{fitz.version[0]}"
    )

except ImportError:

    raise ImportError(
        "PyMuPDF is not installed. "
        "Install the `pymupdf` package on the Databricks "
        "environment before continuing."
    )

In [0]:
# ============================================================
# Helper: infer report year from filename
# ============================================================
#
# This is ingestion-level metadata only.
#
# Later, 04_document_metadata will validate/enrich metadata
# against the document content and manifest.
# ============================================================

def extract_report_year(filename: str):
    """
    Extract a four-digit GEP edition year from a filename.

    Example:
        GEP-Jan-2025.pdf -> 2025
    """

    match = re.search(
        r"\b(2022|2023|2024|2025|2026)\b",
        filename
    )

    if match:
        return int(match.group(1))

    return None

In [0]:
# ============================================================
# Helper: create stable document identifier
# ============================================================

def create_document_id(report_year: int):
    """
    Create a stable human-readable document ID.
    """

    return f"gep_{report_year}_01"

In [0]:
# ============================================================
# Parse one PDF into page-level records
# ============================================================
#
# IMPORTANT:
#
# We preserve:
#   - page number
#   - raw page text
#   - text block count
#   - page dimensions
#
# We are NOT cleaning or chunking here.
#
# Keeping parsing separate makes the pipeline reproducible:
#
# raw PDF -> parsed pages -> cleaned pages -> chunks
# ============================================================

def parse_pdf(pdf_path: str, filename: str):
    """
    Parse a PDF into page-level records.
    """

    report_year = extract_report_year(filename)

    if report_year is None:
        raise ValueError(
            f"Could not determine report year from {filename}"
        )

    document_id = create_document_id(report_year)

    document = fitz.open(pdf_path)

    parsed_pages = []

    for page_index in range(len(document)):

        page = document[page_index]

        # Page number exposed to users is 1-based.
        page_number = page_index + 1

        # Plain page text.
        raw_text = page.get_text("text")

        # Text blocks preserve more layout information
        # than plain text alone.
        blocks = page.get_text("blocks")

        # Count blocks for diagnostics.
        text_block_count = len(blocks)

        # Detect whether the page has embedded images.
        image_count = len(
            page.get_images(full=True)
        )

        # Stable page identifier.
        page_id = (
            f"{document_id}_page_{page_number:04d}"
        )

        parsed_pages.append({

            "document_id": document_id,

            "page_id": page_id,

            "report_year": report_year,

            "filename": filename,

            "file_path": pdf_path,

            "page_number": page_number,

            "raw_text": raw_text,

            "text_block_count": text_block_count,

            "image_count": image_count,

            "page_width": float(page.rect.width),

            "page_height": float(page.rect.height),

            "character_count": (
                len(raw_text)
                if raw_text is not None
                else 0
            )
        })

    document.close()

    return parsed_pages

In [0]:
# ============================================================
# Convert Databricks DBFS URI to local Volume path
# ============================================================

def to_local_volume_path(path: str) -> str:
    """
    Convert a Databricks DBFS Volume URI into a path that
    standard Python libraries such as PyMuPDF can open.

    Example:

        dbfs:/Volumes/catalog/schema/volume/file.pdf

    becomes:

        /Volumes/catalog/schema/volume/file.pdf
    """

    if path.startswith("dbfs:/Volumes/"):

        return path.replace(
            "dbfs:/Volumes/",
            "/Volumes/",
            1
        )

    return path

In [0]:
# ============================================================
# Parse all GEP PDFs
# ============================================================

all_parsed_pages = []

for file_info in pdf_files:

    print(
        f"Parsing: {file_info.name}"
    )

    # --------------------------------------------------------
    # Convert dbfs:/Volumes/... to /Volumes/...
    # --------------------------------------------------------

    local_pdf_path = to_local_volume_path(
        file_info.path
    )

    print(
        f"Local path: {local_pdf_path}"
    )

    # --------------------------------------------------------
    # Verify Python can actually see the file before parsing
    # --------------------------------------------------------

    if not os.path.exists(local_pdf_path):

        raise FileNotFoundError(
            f"PDF is not accessible through the Volume "
            f"filesystem path: {local_pdf_path}"
        )

    # --------------------------------------------------------
    # Parse PDF
    # --------------------------------------------------------

    parsed_pages = parse_pdf(
        pdf_path=local_pdf_path,
        filename=file_info.name
    )

    print(
        f"  Pages parsed: {len(parsed_pages):,}"
    )

    all_parsed_pages.extend(
        parsed_pages
    )


print("=" * 70)

print(
    f"Total pages parsed: "
    f"{len(all_parsed_pages):,}"
)

In [0]:
# ============================================================
# Define parsed-page schema
# ============================================================
#
# Explicit schemas are preferable to inference for production
# pipelines and avoid issues when fields happen to be NULL.
# ============================================================

parsed_page_schema = T.StructType([

    T.StructField(
        "document_id",
        T.StringType(),
        False
    ),

    T.StructField(
        "page_id",
        T.StringType(),
        False
    ),

    T.StructField(
        "report_year",
        T.IntegerType(),
        False
    ),

    T.StructField(
        "filename",
        T.StringType(),
        False
    ),

    T.StructField(
        "file_path",
        T.StringType(),
        False
    ),

    T.StructField(
        "page_number",
        T.IntegerType(),
        False
    ),

    T.StructField(
        "raw_text",
        T.StringType(),
        True
    ),

    T.StructField(
        "text_block_count",
        T.IntegerType(),
        True
    ),

    T.StructField(
        "image_count",
        T.IntegerType(),
        True
    ),

    T.StructField(
        "page_width",
        T.DoubleType(),
        True
    ),

    T.StructField(
        "page_height",
        T.DoubleType(),
        True
    ),

    T.StructField(
        "character_count",
        T.IntegerType(),
        False
    )
])

In [0]:
# ============================================================
# Create parsed page DataFrame
# ============================================================

parsed_pages_df = (
    spark.createDataFrame(
        all_parsed_pages,
        schema=parsed_page_schema
    )

    .withColumn(
        "parsed_at",
        F.current_timestamp()
    )
)

print(
    f"Parsed page records: "
    f"{parsed_pages_df.count():,}"
)

In [0]:
# ============================================================
# Validate parsed document coverage
# ============================================================

document_stats_df = (
    parsed_pages_df

    .groupBy(
        "document_id",
        "report_year",
        "filename"
    )

    .agg(
        F.count("*").alias(
            "page_count"
        ),

        F.sum("character_count").alias(
            "total_characters"
        ),

        F.sum("image_count").alias(
            "total_images"
        )
    )

    .orderBy(
        "report_year"
    )
)

display(document_stats_df)

In [0]:
# ============================================================
# Validate expected report editions
# ============================================================

parsed_years = {
    row["report_year"]
    for row in (
        parsed_pages_df
        .select("report_year")
        .distinct()
        .collect()
    )
}

expected_years = set(
    EXPECTED_REPORT_YEARS
)

print(
    f"Expected years: {sorted(expected_years)}"
)

print(
    f"Parsed years:   {sorted(parsed_years)}"
)


if parsed_years != expected_years:
    raise RuntimeError(
        "Parsed GEP report years do not match expected editions."
    )

print(
    "GEP edition validation passed."
)

In [0]:
# ============================================================
# Inspect pages with little or no extracted text
# ============================================================
#
# This helps identify:
#
#   - cover pages
#   - image-heavy pages
#   - charts
#   - potential parsing failures
#
# We do NOT automatically OCR these pages.
# ============================================================

low_text_pages_df = (
    parsed_pages_df

    .filter(
        F.col("character_count") < 100
    )

    .select(
        "document_id",
        "report_year",
        "page_number",
        "character_count",
        "image_count"
    )

    .orderBy(
        "report_year",
        "page_number"
    )
)

display(low_text_pages_df)

In [0]:
# ============================================================
# Parsing quality validation
# ============================================================
#
# Detect pages where text extraction may have failed or where
# the page may be primarily visual.
# ============================================================

parsing_quality_df = (
    parsed_pages_df
    .withColumn(
        "parsing_status",
        F.when(
            F.col("character_count") == 0,
            "NO_TEXT"
        )
        .when(
            (F.col("character_count") < 100)
            & (F.col("image_count") > 0),
            "VISUAL_HEAVY"
        )
        .when(
            F.col("character_count") < 100,
            "LOW_TEXT"
        )
        .otherwise(
            "TEXT_OK"
        )
    )
)

display(
    parsing_quality_df
    .groupBy(
        "report_year",
        "parsing_status"
    )
    .count()
    .orderBy(
        "report_year",
        "parsing_status"
    )
)

In [0]:
# ============================================================
# Inspect pages that may require special handling
# ============================================================

display(
    parsing_quality_df
    .filter(
        F.col("parsing_status") != "TEXT_OK"
    )
    .select(
        "report_year",
        "page_number",
        "character_count",
        "image_count",
        "parsing_status",
        "raw_text"
    )
    .orderBy(
        "report_year",
        "page_number"
    )
)

In [0]:
# ============================================================
# Add parsing-quality status
# ============================================================
#
# We persist this instead of using it only for validation.
#
# Later stages can use it to:
#   - exclude genuinely empty pages from text chunking
#   - identify pages for table/figure extraction
#   - selectively route visual-heavy pages to a vision model
#   - monitor document parsing quality
# ============================================================

parsed_pages_df = (
    parsed_pages_df

    .withColumn(
        "parsing_status",

        F.when(
            F.col("character_count") == 0,
            F.lit("NO_TEXT")
        )

        .when(
            (F.col("character_count") < 100)
            & (F.col("image_count") > 0),
            F.lit("VISUAL_HEAVY")
        )

        .when(
            F.col("character_count") < 100,
            F.lit("LOW_TEXT")
        )

        .otherwise(
            F.lit("TEXT_OK")
        )
    )
)

In [0]:
# ============================================================
# Persist page-level parsed documents
# ============================================================

(
    parsed_pages_df

    .write

    .format("delta")

    .mode("overwrite")

    .option(
        "overwriteSchema",
        "true"
    )

    .saveAsTable(
        TARGET_TABLE
    )
)

print(
    f"Saved parsed pages to: "
    f"{TARGET_TABLE}"
)